# Signal Processing for Machine Learning

[![Open In
Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lasseufpa/dsp2026/blob/main/guided_projects/proj1_waveforms_classification.ipynb)

The project is about signal classification using machine learning (ML). Given a dataset and classification problem, the goal is to find a configuration for the frontend DSP stage that minimizes data storage requirements (creating a new version of the dataset), computational cost of the ML stage, and maximizes accuracy.

The designer is allowed to work with two degrees of freedom:

1) decimation factor
2) design and use of quantizers (uniform or non-uniform)

It is prohibited to change the ML stage (you cannot change the classifier).

Signal classification pipeline (e.g., with pathology / without pathology):
1) data representation optimization (decimation, bit-depth reduction) → 2) classification → 3) feedback loop to the optimization stage with analysis of the impact on classification performance → 4) search for the configuration that minimizes data storage requirements, computational cost, and maximizes accuracy.

> If you have just clicked the Google Colab Badge from GitHub, it not possible to save changes. Go to ``File > Save a copy in Drive`` to make a copy of this Jupyter notebook in your Google Drive storage, where changes can be saved, or ``File > Download > Download .ipynb`` to download this notebook to your computer.

### EEG task (PhysioNet)

You are given an intentionally oversampled EEG dataset and must recover a robust 200 Hz representation.
Your objective is to resample correctly and improve classification performance.

#### Dataset Summary
- EEG channels: 4 channels (plus sample index column)
- Original sampling rate: 200 Hz

Two challenge tracks are generated with the same folder/file naming structure as the original dataset:
- `auditory-eeg-challenge/easy/Filtered_Data`: clean oversampled version (400 Hz)
- `auditory-eeg-challenge/hard/Filtered_Data`: oversampled version (400 Hz) with additional high-frequency nuisance content

The hard track contains controlled high-frequency content. If downsampling is
done incorrectly (for example, naive slicing), aliasing can fold nuisance energy
into lower frequencies and hurt model performance. If downsampling is done
correctly (anti-alias filtering before decimation), performance should improve.

### Install necessary packages

In [ ]:
!pip install numpy pandas scipy scikit-learn cupy-cuda12x

### Import packages

In [ ]:
import os
import urllib.request
import zipfile
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report

In [3]:
url = "https://nextcloud.lasseufpa.org/s/iEwLCZ6PtsTY3Ja/download/auditory-eeg-challenge.zip"
zip_name = os.path.basename(url)
out_dir = os.getcwd()

print(f"Downloading {url} to {zip_name}...")
urllib.request.urlretrieve(url, zip_name)
print("Download complete. Extracting...")
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(out_dir)
print(f"Extracted to {out_dir}")

Download complete. Extracting...
Extracted to /home/joao/codes/dsp2026/guided_projects


In [4]:
DATASET_PATH = "./auditory-eeg-challenge/hard/Filtered_Data"

In [5]:
import re
from scipy.signal import welch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report


def parse_experiment_from_filename(file_name: str) -> str:
    match = re.search(r"_ex(\d+)", file_name)
    return match.group(1) if match else "unknown"


def extract_window_features(segment: np.ndarray, fs: int = 200) -> np.ndarray:
    # segment shape: [window_samples, channels]
    feature_vector = []
    bands = {
        "delta": (1.0, 4.0),
        "theta": (4.0, 8.0),
        "alpha": (8.0, 13.0),
        "beta": (13.0, 30.0),
        "gamma": (30.0, 40.0),
    }

    for ch in range(segment.shape[1]):
        x = segment[:, ch].astype(float)

        # Time-domain features
        mean = np.mean(x)
        std = np.std(x)
        rms = np.sqrt(np.mean(x**2))
        line_length = np.sum(np.abs(np.diff(x)))
        zcr = np.mean(np.diff(np.signbit(x)) != 0)

        # Hjorth parameters
        dx = np.diff(x)
        ddx = np.diff(dx)
        var_x = np.var(x) + 1e-12
        var_dx = np.var(dx) + 1e-12
        var_ddx = np.var(ddx) + 1e-12
        hjorth_activity = var_x
        hjorth_mobility = np.sqrt(var_dx / var_x)
        hjorth_complexity = np.sqrt(var_ddx / var_dx) / hjorth_mobility

        feature_vector.extend([
            mean,
            std,
            rms,
            line_length,
            zcr,
            hjorth_activity,
            hjorth_mobility,
            hjorth_complexity,
        ])

        # Frequency-domain features
        freqs, psd = welch(x, fs=fs, nperseg=min(256, len(x)))
        total_power = np.trapezoid(psd, freqs) + 1e-12

        for _, (f_low, f_high) in bands.items():
            idx = (freqs >= f_low) & (freqs < f_high)
            band_power = np.trapezoid(psd[idx], freqs[idx]) if np.any(idx) else 0.0
            rel_power = band_power / total_power
            feature_vector.extend([band_power, rel_power])

    return np.asarray(feature_vector, dtype=np.float32)


def load_dataset_with_features(dataset_path: str, window: int = 400, step: int = 200, fs: int = 200):
    X, y, groups, experiments, file_names = [], [], [], [], []

    for root, _, files in os.walk(dataset_path):
        for file in sorted(files):
            if not file.endswith(".csv"):
                continue

            full_path = os.path.join(root, file)
            recording_id = os.path.splitext(file)[0]
            subject = file.split("_")[0]  # e.g., s01
            experiment = parse_experiment_from_filename(file)

            df = pd.read_csv(full_path)
            if df.shape[1] < 5:
                continue

            eeg = df.iloc[:, 1:5].to_numpy(dtype=float)  # 4 channels
            if len(eeg) < window:
                continue

            # Per-recording per-channel normalization to reduce session scale effects
            eeg = (eeg - eeg.mean(axis=0, keepdims=True)) / (eeg.std(axis=0, keepdims=True) + 1e-8)

            for i in range(0, len(eeg) - window + 1, step):
                segment = eeg[i : i + window]
                X.append(extract_window_features(segment, fs=fs))
                y.append(subject)
                groups.append(recording_id)
                experiments.append(experiment)
                file_names.append(file)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y),
        np.asarray(groups),
        np.asarray(experiments),
        np.asarray(file_names),
    )

In [6]:
window = 400
step = 200

Xf, yf, groups, exp_ids, files = load_dataset_with_features(
    DATASET_PATH,
    window=window,
    step=step,
    fs=200,
)

if len(Xf) == 0:
    raise ValueError(
        f"No feature windows were loaded from {DATASET_PATH}. "
        "Check dataset path and run preprocessing cells first."
    )

print(f"Feature matrix shape: {Xf.shape}")
print(f"Unique subjects: {len(np.unique(yf))}")
print(f"Unique recordings (groups): {len(np.unique(groups))}")

# Leakage-safe split: entire recordings are kept in train or test
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(Xf, yf, groups=groups))

X_train, X_test = Xf[train_idx], Xf[test_idx]
y_train, y_test = yf[train_idx], yf[test_idx]
g_train, g_test = groups[train_idx], groups[test_idx]
f_test = files[test_idx]

print(f"Train windows: {len(X_train)} | Test windows: {len(X_test)}")
print(f"Train recordings: {len(np.unique(g_train))} | Test recordings: {len(np.unique(g_test))}")

AttributeError: module 'numpy' has no attribute 'trapezoid'

### XGBoost pipeline (GPU if available)

This section trains an XGBoost classifier on the leakage-safe split.
It tries GPU acceleration first and automatically falls back to CPU if GPU is unavailable.

In [ ]:
from collections import Counter
from time import perf_counter
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder


label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)

xgb_common = dict(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
)

xgb_backend = "GPU"
xgb_prediction_device = "GPU"
start = perf_counter()

try:
    xgb_clf = XGBClassifier(
        **xgb_common,
        tree_method="hist",
        device="cuda",
    )
    xgb_clf.fit(X_train, y_train_enc)
except Exception as gpu_error:
    xgb_backend = "CPU"
    xgb_prediction_device = "CPU"
    print(f"GPU not available or failed for XGBoost: {gpu_error}")
    xgb_clf = XGBClassifier(
        **xgb_common,
        tree_method="hist",
        device="cpu",
        n_jobs=-1,
    )
    xgb_clf.fit(X_train, y_train_enc)

xgb_train_seconds = perf_counter() - start

# Avoid device mismatch warnings by matching prediction data/device.
if xgb_backend == "GPU":
    try:
        import cupy as cp

        X_test_gpu = cp.asarray(X_test)
        xgb_proba = xgb_clf.get_booster().inplace_predict(X_test_gpu)
        xgb_proba = cp.asnumpy(xgb_proba)
    except Exception as pred_gpu_error:
        xgb_prediction_device = "CPU"
        print(f"GPU prediction fallback to CPU: {pred_gpu_error}")
        xgb_clf.set_params(device="cpu")
        xgb_proba = xgb_clf.predict_proba(X_test)
else:
    xgb_proba = xgb_clf.predict_proba(X_test)

xgb_y_pred_enc = np.argmax(xgb_proba, axis=1).astype(int)
xgb_y_pred = label_encoder.inverse_transform(xgb_y_pred_enc)
xgb_win_acc = accuracy_score(y_test, xgb_y_pred)
xgb_win_bal = balanced_accuracy_score(y_test, xgb_y_pred)

xgb_class_labels = label_encoder.classes_

xgb_recording_true = {}
xgb_recording_pred = {}

for rec in np.unique(g_test):
    rec_mask = g_test == rec
    rec_proba = xgb_proba[rec_mask].mean(axis=0)
    rec_pred = xgb_class_labels[np.argmax(rec_proba)]
    rec_true = Counter(y_test[rec_mask]).most_common(1)[0][0]
    xgb_recording_true[rec] = rec_true
    xgb_recording_pred[rec] = rec_pred

xgb_y_true_rec = np.array([xgb_recording_true[r] for r in xgb_recording_true])
xgb_y_pred_rec = np.array([xgb_recording_pred[r] for r in xgb_recording_true])

xgb_rec_acc = accuracy_score(xgb_y_true_rec, xgb_y_pred_rec)
xgb_rec_bal = balanced_accuracy_score(xgb_y_true_rec, xgb_y_pred_rec)

# Generic aliases used by downstream diagnostics
model_name = "XGBoost"
model_y_pred = xgb_y_pred
model_recording_true = xgb_recording_true
model_recording_pred = xgb_recording_pred

print(f"Model used: {model_name}")
print(f"XGBoost backend used: {xgb_backend}")
print(f"XGBoost prediction device: {xgb_prediction_device}")
print(f"XGBoost fit time (s): {xgb_train_seconds:.2f}")

print("\nWindow-level metrics")
print(f"Accuracy: {xgb_win_acc:.4f}")
print(f"Balanced accuracy: {xgb_win_bal:.4f}")

print("\nRecording-level metrics")
print(f"Accuracy: {xgb_rec_acc:.4f}")
print(f"Balanced accuracy: {xgb_rec_bal:.4f}")

print("\nPer-subject report (window-level)")
print(classification_report(y_test, model_y_pred, zero_division=0))

In [ ]:
# Per-experiment diagnostics for the current model outputs
from sklearn.metrics import recall_score


def safe_balanced_accuracy(y_true_subset, y_pred_subset):
    labels = np.unique(np.concatenate([y_true_subset, y_pred_subset]))
    return recall_score(y_true_subset, y_pred_subset, labels=labels, average="macro", zero_division=0)


if "model_y_pred" not in globals():
    raise RuntimeError("Run the XGBoost training cell first.")

exp_test = exp_ids[test_idx]

print(f"Per-experiment metrics (window-level) - {model_name}")
for ex in sorted(np.unique(exp_test)):
    m = exp_test == ex
    if np.sum(m) < 20:
        continue
    ex_acc = accuracy_score(y_test[m], model_y_pred[m])
    ex_bal = safe_balanced_accuracy(y_test[m], model_y_pred[m])
    print(f"ex{ex}: windows={np.sum(m):4d} | acc={ex_acc:.4f} | bal_acc={ex_bal:.4f}")

print(f"\nPer-experiment metrics (recording-level) - {model_name}")
for ex in sorted(np.unique(exp_test)):
    recs_ex = np.unique(g_test[exp_test == ex])
    if len(recs_ex) == 0:
        continue

    y_true_ex = []
    y_pred_ex = []
    for rec in recs_ex:
        y_true_ex.append(model_recording_true[rec])
        y_pred_ex.append(model_recording_pred[rec])

    y_true_ex = np.asarray(y_true_ex)
    y_pred_ex = np.asarray(y_pred_ex)
    ex_acc = accuracy_score(y_true_ex, y_pred_ex)
    ex_bal = safe_balanced_accuracy(y_true_ex, y_pred_ex)
    print(f"ex{ex}: recordings={len(recs_ex):3d} | acc={ex_acc:.4f} | bal_acc={ex_bal:.4f}")